# Batched Throughput: does the compute-saving claim survive standard practice?

## Why this run exists

`locked-results.md` §3k measured the pipeline **one sequence at a time** — which is how every
notebook in this project ran ESMFold, but is *not* how anyone deploys it. §3k-AUDIT tried to
pre-empt the objection with an analytic table, but that table made a **pessimistic and unjustified
assumption**: it divided fold time by a batching speedup while holding generation time *fixed*.

In reality you would batch **both** stages. And the two stages have opposite memory profiles —
autoregressive generation is memory-light and batches easily, while ESMFold is memory-heavy and
batches poorly under padding. So batching could plausibly move the fold share in **either**
direction:

- If generation speeds up more than folding → **fold share rises → triage becomes MORE valuable.**
- If folding speeds up more than generation → fold share falls → triage becomes less valuable.

§3k-AUDIT assumed the second without testing it. This notebook measures which is true.

## What it measures

1. **ESMFold throughput** at batch sizes 1/2/4/8, with automatic OOM back-off (T4 has 16 GB and
   folding is memory-hungry; the largest feasible batch is itself a reportable number).
2. **The same, with length-bucketing** — sorting by length before batching so padding waste is
   minimised. This is what a competent practitioner would actually do, and naive batching would
   understate batched folding's true speed.
3. **ProtGPT2 generation throughput** at batch sizes 1/2/4/8/16, using `padding_side="left"`
   (required for correct batched autoregressive generation — right-padding silently corrupts it).
4. **A validity check that matters beyond this notebook:** does batched folding return the *same*
   pLDDT per sequence as unbatched? Every locked result in this project used unbatched folding. If
   batching changed the numbers, that would be a project-wide problem, not just a throughput note.
5. **Recomputed triage savings** under each regime, replacing §3k-AUDIT's assumed table with
   measured ones.

## The padding-mask correctness issue, handled explicitly

ESMFold returns per-residue pLDDT shaped `(batch, seq_len, atoms)`. With a batch, shorter sequences
are **padded**, and naively averaging over the whole tensor would mix real residues with padding —
producing wrong pLDDT that would silently corrupt both the validity check and any collapse label.
Every pLDDT here is averaged **only over each sequence's true length**, taken from the attention
mask. This is the single most likely place for a batched-folding implementation to go quietly wrong.

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect ~35–55 minutes.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc, time, math, urllib.request

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding

torch.manual_seed(2024); np.random.seed(2024)
device = "cuda" if torch.cuda.is_available() else "cpu"

N_SEQS = 64                      # fixed set reused across every batch size
GEN_BATCHES = [1, 2, 4, 8, 16]
FOLD_BATCHES = [1, 2, 4, 8]
MAX_LEN = 50
WARMUP = 2

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

def clear_gpu():
    gc.collect(); torch.cuda.empty_cache()

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

class Timer:
    def __enter__(self): sync(); self.t0 = time.perf_counter(); return self
    def __exit__(self, *a): sync(); self.dt = time.perf_counter() - self.t0

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")
print(f"\nPlan: {N_SEQS} sequences | gen batches {GEN_BATCHES} | fold batches {FOLD_BATCHES}")


CUDA: True
GPU: Tesla T4
Total VRAM: 15.6 GB

Plan: 64 sequences | gen batches [1, 2, 4, 8, 16] | fold batches [1, 2, 4, 8]


In [2]:
# --- Build the fixed sequence set. Generated fresh with the project's standard recipe so the
#     length distribution matches everything else (§3k saw mean 58, range 10-210). ---

UNIPROT = ["P0CG48","P00720","P02144","P42212","P01308","P61823",
           "P00648","P99999","P69905","P68871","P00698","P00441"]
FALLBACK = ["NLYIQWLKDGGPSSGRPPPS","LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF","GSQIGAKNTGQVQLNLLAL",
            "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
            "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
            "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV"]

refs = []
for acc in UNIPROT:
    try:
        with urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=10) as r:
            lines = [l for l in r.read().decode("utf-8").strip().split("\n") if l]
        s = "".join(lines[1:])
        if len(s) >= 20: refs.append(s)
    except Exception as e:
        print(f"  skip {acc}: {e}")
if not refs:
    print("!! UniProt fetch failed (Internet toggle OFF?). Using hardcoded fallback.")
    refs = FALLBACK
print(f"{len(refs)} source proteins.")

rng = np.random.RandomState(909)
prefixes = []
for i in range(N_SEQS):
    s = refs[i % len(refs)]
    pl = rng.randint(10, 16)
    st = rng.randint(0, max(1, len(s) - pl))
    prefixes.append(s[st:st + pl])

print(f"Loading ProtGPT2 on {device}...")
tok = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
gen_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device).eval()

# Build the fixed sequence set once, unbatched, so every later timing folds IDENTICAL inputs.
torch.manual_seed(909)
seqs = []
for p in prefixes:
    inp = tok(p, return_tensors="pt").to(device)
    with torch.no_grad():
        o = gen_model.generate(**inp, max_length=MAX_LEN, do_sample=True,
                               temperature=1.2, pad_token_id=tok.eos_token_id)
    s = tok.decode(o[0], skip_special_tokens=True).replace(" ", "")
    seqs.append("".join(a for a in s if a in VALID_AA))
clear_gpu()

lens = [len(s) for s in seqs]
print(f"\nFixed set: {len(seqs)} sequences, length mean {np.mean(lens):.1f}, "
      f"range {min(lens)}-{max(lens)}, sd {np.std(lens):.1f}")
print(f"(§3k reference: mean 58, range 10-210 — similar distribution confirms comparability)")


12 source proteins.
Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fixed set: 64 sequences, length mean 51.9, range 10-142, sd 44.7
(§3k reference: mean 58, range 10-210 — similar distribution confirms comparability)


In [3]:
# --- GENERATION throughput vs batch size. padding_side='left' is REQUIRED: with right-padding,
#     HF continues from pad tokens and batched generation silently produces garbage. ---

tok.padding_side = "left"

def timed_generate(prompts, batch_size):
    out_seqs, total = [], 0.0
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        enc = tok(chunk, return_tensors="pt", padding=True).to(device)
        with Timer() as t:
            with torch.no_grad():
                o = gen_model.generate(**enc, max_length=MAX_LEN, do_sample=True,
                                       temperature=1.2, pad_token_id=tok.eos_token_id)
        total += t.dt
        for row in o:
            out_seqs.append(tok.decode(row, skip_special_tokens=True).replace(" ", ""))
    return out_seqs, total

print("Warm-up...")
for _ in range(WARMUP):
    timed_generate(prefixes[:4], 4)

gen_rows = []
print(f"\n{'batch':>6s} {'total s':>9s} {'s/seq':>9s} {'speedup':>9s} {'peak GB':>9s}")
print("-" * 48)
base_gen = None
for bs in GEN_BATCHES:
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    try:
        torch.manual_seed(909)
        _, total = timed_generate(prefixes, bs)
        per = total / len(prefixes)
        if base_gen is None: base_gen = per
        peak = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
        gen_rows.append({"batch": bs, "total_s": total, "s_per_seq": per,
                         "speedup": base_gen / per, "peak_gb": peak})
        print(f"{bs:6d} {total:9.2f} {per:9.4f} {base_gen/per:8.2f}x {peak:9.2f}")
    except RuntimeError as e:
        print(f"{bs:6d}   OOM/error: {str(e)[:50]}")
        clear_gpu()
    clear_gpu()

del gen_model; clear_gpu()
gen_df = pd.DataFrame(gen_rows)


Warm-up...

 batch   total s     s/seq   speedup   peak GB
------------------------------------------------
     1     30.51    0.4767     1.00x      3.50
     2     22.10    0.3453     1.38x      3.52
     4     16.16    0.2525     1.89x      3.55
     8     10.58    0.1653     2.88x      3.64
    16      5.26    0.0822     5.80x      3.80


In [4]:
# --- FOLDING throughput vs batch size, with the padding-mask fix applied to every pLDDT.
#     Two orderings: as-generated (naive) and length-sorted (bucketed), because padding waste is
#     the dominant cost in naive batched folding and a real pipeline would sort. ---

print("Loading ESMFold...")
esm_tok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
esm = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1",
                                           low_cpu_mem_usage=True).to(device).eval()

def fold_batch_timed(batch_seqs):
    # Returns (per-sequence plddt list, elapsed). pLDDT is averaged ONLY over each sequence's
    # true residues using the attention mask -- averaging over padding would corrupt the values.
    enc = esm_tok(batch_seqs, return_tensors="pt", add_special_tokens=False, padding=True).to(device)
    with Timer() as t:
        with torch.no_grad():
            out = esm(**enc)
    plddt = out.plddt.cpu().numpy()          # (B, L, atoms) or (B, L)
    mask = enc["attention_mask"].cpu().numpy()   # (B, L)
    per_seq = []
    for b in range(len(batch_seqs)):
        L = int(mask[b].sum())
        p = plddt[b, :L]
        v = float(np.mean(p))
        per_seq.append(v * 100.0 if v <= 1.5 else v)
    return per_seq, t.dt

def run_fold_sweep(seq_list, label):
    rows, base = [], None
    order = list(range(len(seq_list)))
    print(f"\n--- {label} ---")
    print(f"{'batch':>6s} {'total s':>9s} {'s/seq':>9s} {'speedup':>9s} {'peak GB':>9s} {'status':>10s}")
    print("-" * 60)
    plddt_by_batch = {}
    for bs in FOLD_BATCHES:
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        total, vals, ok = 0.0, [None] * len(seq_list), True
        try:
            for i in range(0, len(seq_list), bs):
                idx = order[i:i + bs]
                chunk = [seq_list[j] for j in idx]
                ps, dt = fold_batch_timed(chunk)
                total += dt
                for j, v in zip(idx, ps):
                    vals[j] = v
        except RuntimeError as e:
            ok = False
            print(f"{bs:6d}   OOM at this batch size — {str(e)[:40]}")
            clear_gpu()
        if not ok:
            continue
        per = total / len(seq_list)
        if base is None: base = per
        peak = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
        rows.append({"ordering": label, "batch": bs, "total_s": total, "s_per_seq": per,
                     "speedup": base / per, "peak_gb": peak})
        plddt_by_batch[bs] = vals
        print(f"{bs:6d} {total:9.2f} {per:9.4f} {base/per:8.2f}x {peak:9.2f} {'ok':>10s}")
        clear_gpu()
    return rows, plddt_by_batch

print("Warm-up...")
for _ in range(WARMUP):
    fold_batch_timed(seqs[:2])

naive_rows, naive_plddt = run_fold_sweep(seqs, "naive order")

order_sorted = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
seqs_sorted = [seqs[i] for i in order_sorted]
sorted_rows, sorted_plddt = run_fold_sweep(seqs_sorted, "length-sorted")

fold_df = pd.DataFrame(naive_rows + sorted_rows)
del esm; clear_gpu()


Loading ESMFold...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warm-up...

--- naive order ---
 batch   total s     s/seq   speedup   peak GB     status
------------------------------------------------------------
     1    132.87    2.0761     1.00x     14.47         ok
     2    142.66    2.2291     0.93x     14.66         ok
     4    175.97    2.7496     0.76x     15.06         ok
     8   OOM at this batch size — CUDA out of memory. Tried to allocate 22

--- length-sorted ---
 batch   total s     s/seq   speedup   peak GB     status
------------------------------------------------------------
     1    130.70    2.0421     1.00x     14.47         ok
     2    105.54    1.6491     1.24x     14.66         ok
     4     97.93    1.5302     1.33x     15.06         ok
     8   OOM at this batch size — CUDA out of memory. Tried to allocate 22


In [5]:
# --- VALIDITY CHECK: does batching change pLDDT? Every locked result used batch=1. If batched
#     folding disagrees, that is a project-wide issue, not a throughput footnote. ---

print("=" * 78)
print("VALIDITY — batched vs unbatched pLDDT on the SAME sequences")
print("=" * 78)

ref = naive_plddt.get(1)
if ref is None:
    print("No batch=1 reference available; cannot run validity check.")
else:
    print(f"{'batch':>6s} {'n':>5s} {'mean |diff|':>12s} {'max |diff|':>11s} {'corr':>8s} "
          f"{'label flips':>12s}")
    print("-" * 60)
    val_rows = []
    for bs, vals in sorted(naive_plddt.items()):
        if bs == 1: continue
        pairs = [(a, b) for a, b in zip(ref, vals) if a is not None and b is not None]
        if not pairs: continue
        a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])
        diff = np.abs(a - b)
        flips = int(np.sum((a < 60) != (b < 60)))
        corr = float(np.corrcoef(a, b)[0, 1]) if len(a) > 2 else float("nan")
        val_rows.append({"batch": bs, "mean_abs_diff": float(diff.mean()),
                         "max_abs_diff": float(diff.max()), "corr": corr, "label_flips": flips})
        print(f"{bs:6d} {len(a):5d} {diff.mean():12.3f} {diff.max():11.3f} {corr:8.4f} "
              f"{flips:9d}/{len(a)}")

    worst = max((r["mean_abs_diff"] for r in val_rows), default=0.0)
    total_flips = sum(r["label_flips"] for r in val_rows)
    print()
    if worst < 1.0 and total_flips == 0:
        print("  ==> Batching does NOT change pLDDT (mean abs diff < 1 point, zero collapse-label")
        print("      flips). Every previously-locked unbatched result stands unaffected, and")
        print("      batched timings are directly comparable to unbatched ones.")
    elif total_flips > 0:
        print(f"  ==> WARNING: {total_flips} collapse-label flips between batched and unbatched.")
        print("      Investigate before using batched folding for anything but timing. Most likely")
        print("      cause would be a padding-mask error, though this notebook masks explicitly.")
    else:
        print(f"  ==> Small numeric drift (mean {worst:.2f} pLDDT points) but no label flips.")
        print("      Acceptable for timing; note it if batched pLDDT values are ever quoted.")
    pd.DataFrame(val_rows).to_csv("batched_validity_check.csv", index=False)


VALIDITY — batched vs unbatched pLDDT on the SAME sequences
 batch     n  mean |diff|  max |diff|     corr  label flips
------------------------------------------------------------
     2    64        0.000       0.003   1.0000         0/64
     4    64        0.001       0.051   1.0000         0/64

  ==> Batching does NOT change pLDDT (mean abs diff < 1 point, zero collapse-label
      flips). Every previously-locked unbatched result stands unaffected, and
      batched timings are directly comparable to unbatched ones.


In [6]:
# --- THE ANSWER: fold share and triage savings under each batching regime. ---

T_GEN_1 = float(gen_df[gen_df.batch == 1].s_per_seq.iloc[0])
best_gen = gen_df.loc[gen_df.s_per_seq.idxmin()]
naive_f = fold_df[fold_df.ordering == "naive order"]
sorted_f = fold_df[fold_df.ordering == "length-sorted"]
T_FOLD_1 = float(naive_f[naive_f.batch == 1].s_per_seq.iloc[0])
best_fold_naive = naive_f.loc[naive_f.s_per_seq.idxmin()]
best_fold_sorted = sorted_f.loc[sorted_f.s_per_seq.idxmin()] if len(sorted_f) else best_fold_naive

print("=" * 92)
print("MEASURED THROUGHPUT")
print("=" * 92)
print(f"Generation : batch 1 -> {T_GEN_1:.4f}s/seq | best batch {int(best_gen.batch)} -> "
      f"{best_gen.s_per_seq:.4f}s/seq ({best_gen.speedup:.2f}x)")
print(f"Folding    : batch 1 -> {T_FOLD_1:.4f}s/seq | best naive batch "
      f"{int(best_fold_naive.batch)} -> {best_fold_naive.s_per_seq:.4f}s/seq "
      f"({best_fold_naive.speedup:.2f}x)")
print(f"Folding    : best LENGTH-SORTED batch {int(best_fold_sorted.batch)} -> "
      f"{best_fold_sorted.s_per_seq:.4f}s/seq ({best_fold_sorted.speedup:.2f}x)")
print()
print(f"*** Which stage benefits more from batching? ***")
print(f"    generation speedup {best_gen.speedup:.2f}x  vs  folding speedup "
      f"{best_fold_sorted.speedup:.2f}x (length-sorted)")
if best_gen.speedup > best_fold_sorted.speedup:
    print("    -> GENERATION benefits more. Fold share RISES under batching, so triage becomes")
    print("       MORE valuable, not less. §3k-AUDIT's analytic table was pessimistic.")
else:
    print("    -> FOLDING benefits more. Fold share falls under batching; §3k-AUDIT's direction")
    print("       was right, and the measured magnitude below replaces its assumed one.")

T_CLF = 0.0271   # §3k measured, layer-30 forward pass at 30 residues

print()
print("=" * 92)
print("TRIAGE SAVINGS UNDER EACH REGIME  (abort@30 residues, fold top b%)")
print("=" * 92)
print(f"{'regime':>26s} {'fold share':>11s} {'b=70%':>8s} {'b=50%':>8s} {'b=30%':>8s}")
print("-" * 68)

regimes = [
    ("unbatched (§3k)", T_GEN_1, T_FOLD_1),
    (f"gen b{int(best_gen.batch)} / fold b1", float(best_gen.s_per_seq), T_FOLD_1),
    (f"gen b1 / fold b{int(best_fold_sorted.batch)} sorted", T_GEN_1, float(best_fold_sorted.s_per_seq)),
    (f"BOTH batched (sorted)", float(best_gen.s_per_seq), float(best_fold_sorted.s_per_seq)),
]
rows_out = []
for name, tg, tf in regimes:
    base = tg + tf
    share = tf / base
    line = f"{name:>26s} {share:>10.0%}"
    for b in (0.70, 0.50, 0.30):
        # abort at 30 of ~47 generated residues -> fraction of generation done before decision
        frac_gen = 30.0 / 47.3
        tg_k = tg * frac_gen
        cost = tg_k + T_CLF + b * ((tg - tg_k) + tf)
        saved = 1 - cost / base
        line += f" {saved:7.1%}"
        rows_out.append({"regime": name, "t_gen_s": tg, "t_fold_s": tf, "fold_share": share,
                         "budget": b, "saved": saved})
    print(line)

print()
print("Read: 'BOTH batched' is the realistic deployment regime and the number the paper should")
print("quote. The unbatched row is retained only for continuity with §3k.")

pd.DataFrame(rows_out).to_csv("batched_triage_savings.csv", index=False)
gen_df.to_csv("batched_generation_throughput.csv", index=False)
fold_df.to_csv("batched_folding_throughput.csv", index=False)
print("\nSaved: batched_{generation,folding}_throughput.csv, batched_triage_savings.csv,")
print("       batched_validity_check.csv")
print()
print("=" * 78)
print("WHAT TO DO WITH THIS")
print("=" * 78)
print("1. Replace §3k-AUDIT's ASSUMED batching table with these MEASURED numbers.")
print("2. Quote the 'BOTH batched (sorted)' row as the headline compute saving — it is the")
print("   regime a practitioner would actually run.")
print("3. If the validity check passed, state explicitly that batching does not alter pLDDT, so")
print("   every unbatched locked result remains valid.")
print("4. Report the largest feasible fold batch size and peak VRAM — that is the practical")
print("   constraint another lab would hit, and it is resource-constrained-discovery content")
print("   (Track 5) in its own right.")


MEASURED THROUGHPUT
Generation : batch 1 -> 0.4767s/seq | best batch 16 -> 0.0822s/seq (5.80x)
Folding    : batch 1 -> 2.0761s/seq | best naive batch 1 -> 2.0761s/seq (1.00x)
Folding    : best LENGTH-SORTED batch 4 -> 1.5302s/seq (1.33x)

*** Which stage benefits more from batching? ***
    generation speedup 5.80x  vs  folding speedup 1.33x (length-sorted)
    -> GENERATION benefits more. Fold share RISES under batching, so triage becomes
       MORE valuable, not less. §3k-AUDIT's analytic table was pessimistic.

TRIAGE SAVINGS UNDER EACH REGIME  (abort@30 residues, fold top b%)
                    regime  fold share    b=70%    b=50%    b=30%
--------------------------------------------------------------------
           unbatched (§3k)        81%   25.4%   43.0%   60.6%
         gen b16 / fold b1        96%   28.0%   47.5%   67.1%
   gen b1 / fold b4 sorted        76%   24.1%   41.1%   58.1%
     BOTH batched (sorted)        95%   27.3%   46.7%   66.1%

Read: 'BOTH batched' is the 